In [1]:
#undef __noinline__

# Speed comparison with multiple threads

In the previous notebooks we only ran kernels with multiple blocks and one thread. This is slow because we are not maximizing the use of a block. Here we can see the speed boost from running a vector add kernel but with multiple threads per block.

In the first cell we can see two global kernels. The first one is the same as before, but the second one has a different index: `blockIdx.x * blockDim.x + threadIdx.x`. This is the standard cuda thread indexing formula that gives every GPU thread a unique global id across the entire grid.

In [2]:
#define N 10000000

__global__ void vector_add_block( int *a, int *b, int *c ) {
     int tid = blockIdx.x;
     if (tid < N)
     c[tid] = a[tid] + b[tid];
}

__global__ void vector_add( int *a, int *b, int *c) {
     int tid = blockIdx.x * blockDim.x + threadIdx.x;
     if (tid < N)
         c[tid] = a[tid] + b[tid];
}

Here we have the standard setup that we have used before.

In [3]:
int v_a[N], v_b[N], v_c[N];
int *dev_a, *dev_b, *dev_c;

// allocate the memory on the GPU
cudaMalloc( (void**)&dev_a, N * sizeof(int) );
cudaMalloc( (void**)&dev_b, N * sizeof(int) );
cudaMalloc( (void**)&dev_c, N * sizeof(int) );

// fill the arrays 'a' and 'b' on the CPU
for (int i=0; i<N; i++) {
    v_a[i] = i;
    v_b[i] = i * i;
}

// copy the arrays 'a' and 'b' to the GPU
cudaMemcpy( dev_a, v_a, N * sizeof(int), cudaMemcpyHostToDevice );
cudaMemcpy( dev_b, v_b, N * sizeof(int), cudaMemcpyHostToDevice );

This cell calls the first kernel with an N number of blocks and one thread for each block. We have already done this. The new code here is that we track the time it takes the kernel to add the numbers using the **cudaEventRecord()** function.

In [4]:
float time_block_method = 0;
float time_thread_method = 0;

cudaEvent_t start, stop;

cudaEventCreate(&start);
cudaEventCreate(&stop);

cudaEventRecord(start);

vector_add_block<<<N, 1>>>(dev_a, dev_b, dev_c); 

cudaEventRecord(stop);

cudaEventSynchronize(stop);

cudaEventElapsedTime(&time_block_method, start, stop);

Next we call the second kernel buth with 256 threads for each block. The number of blocks needed is calculated by this formula: `(N + threadsPerBlock - 1) / threadsPerBlock`

In [5]:
int threadsPerBlock = 256;
int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

cudaEventRecord(start);

vector_add<<<blocksPerGrid, threadsPerBlock>>>(dev_a, dev_b, dev_c); 

cudaEventRecord(stop);
cudaEventSynchronize(stop);
cudaEventElapsedTime(&time_thread_method, start, stop);

In [6]:
// Destroy time events and free allocated memory
cudaEventDestroy(start);
cudaEventDestroy(stop);

cudaFree( dev_a );
cudaFree( dev_b );
cudaFree( dev_c );

This cell here is a CPU only version of the vector addititon. 

In [7]:
#include <iostream>
#include <vector>
#include <chrono>


void add_CPU(const int* a, const int* b, int* c, int n) {
    for (int i = 0; i < n; i++) {
        c[i] = a[i] + b[i];
    }
}

    // Using std::vector for easy memory management on the CPU heap
    std::vector<int> h_a(N), h_b(N), h_c(N);

    // Initialize arrays
    for (int i = 0; i < N; i++) {
        h_a[i] = i;
        h_b[i] = i * 2;
    }

    // Start Timer
    auto cpu_start = std::chrono::high_resolution_clock::now();

    add_CPU(h_a.data(), h_b.data(), h_c.data(), N);

    // Stop Timer
    auto cpu_end = std::chrono::high_resolution_clock::now();

    // Calculate duration in milliseconds
    std::chrono::duration<double, std::milli> duration = cpu_end - cpu_start;

And finaly we print the numbers. We can see that the CPU is faster than the one thread GPU kernel but the 256 thread one is much faster than both.

In [8]:
printf("Time (CPU sort): %f ms\n", duration.count());

printf("Time (1 thread/block): %f ms\n", time_block_method);

printf("Time (256 threads/block): %f ms\n", time_thread_method);

Time (CPU sort): 19.897666 ms
Time (1 thread/block): 43.284351 ms
Time (256 threads/block): 1.766080 ms
